## RAG for Hyderabad Institute of Technology 

In [ ]:
!pip install pypdf
!pip install langchain
!pip install langchain-community
!pip install sentence-transformers


In [ ]:
import sys
print(sys.executable)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader  # Used for loading the pdf
from pathlib import Path 

In [ ]:
knowledge_base=Path(r"C:\Users\agarw\OneDrive\Desktop\RAG project\Knowledge base")

# this will create a path object that will point to  folder

In [ ]:
pdf_files=knowledge_base.glob("*.pdf")

# this will store all the files ending with .pdf into the pdf_files variable 


In [ ]:
documents=[]

In [ ]:
# now we use for loop to deal with pdf one one at a time and also create empty documents list so that later we can add all the pdfs in one place 
#  we convert pdf into the string coz the pyPDFloader expects string 

for pdf in pdf_files:
    loader = PyPDFLoader(str(pdf))
    loaded_pages = loader.load()
    documents.extend(loaded_pages)
    

# only remove the comments when a new pdf is added to the folder

    
    

In [ ]:
len(documents)

In [ ]:
documents

In [ ]:
## To replace the \n coz its causing each word to go to new line and make our text look bad we use this
import re
for doc in documents:
    doc.page_content=re.sub(r"\s+"," ",doc.page_content).strip() 
    
# Here we have replaved the whitespaces with a single space only in the page content of the doc
# Normalize all whitespace (spaces, tabs, newlines)
# into a single space and remove leading/trailing spaces.

In [ ]:
print(documents)

In [ ]:
print(documents)
len(documents)

In [ ]:
print(documents[0].page_content)

In [ ]:
## Perform Chunking on the PDF 

from langchain_text_splitters import RecursiveCharacterTextSplitter # used for chunking 

In [ ]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1300,
    chunk_overlap=450,
    length_function=len,
    separators=["\n\n","\n"," ",""]  #  It is a class and we have made a object for this
)

In [ ]:
chunks = text_splitter.split_documents(documents) # now the dcoument is splitted into smaller smaller chunks 

In [ ]:
len(chunks)

In [ ]:
print(chunks[0])
print("="*100)
print(chunks[1])
print("="*100)
print(chunks[2])
print("="*100)
# print(chunks[])

In [ ]:
print(chunks[0].page_content)


In [ ]:
print(chunks[0].metadata)

In [ ]:
## EMBEDDINGS STARTS HERE 

from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
# pip install sentence-transformers

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
embeddings=embedding_model.embed_documents([chunk.page_content for chunk in chunks])  # embedding done

# We cretaed this just so we can see that how embeddings work and how they look like 
# FAISS creates is own embeddings and store them 


In [ ]:
type(embeddings)


In [ ]:
len(embeddings[0])

In [ ]:
pip install chromadb

In [ ]:
pip install langchain-chroma

In [ ]:
from langchain_community.vectorstores import Chroma

In [ ]:
# UN_Comment only when you want to add new pdf or want to create new database

vector_store=Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="University_RAG",
    persist_directory=r"C:\Users\agarw\OneDrive\Desktop\RAG project\chroma_db"
)
## Vector Database created.

In [ ]:
# use it to get info from the existing database.
vector_store = Chroma(
    persist_directory=r"C:\Users\agarw\OneDrive\Desktop\RAG project\chroma_db",
    embedding_function=embedding_model,
    collection_name="University_RAG"
)

In [ ]:
type(vector_store)

In [ ]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":5,
        "fetch_k":15,
        "lambda_mult":1
    }
)

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
import os

# load_dotenv(r"C:\Users\agarw\OneDrive\Desktop\RAG project\.env")



In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [ ]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [ ]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant for Hyderabad Institute of Technology.

Answer the user's question using ONLY the provided context.

The question may contain multiple parts.
Answer every part that can be answered from the context.

If the context contains multiple policies or multiple values for the same concept, 
explain which policy each value belongs to instead of assuming they refer to the same situation.




Only if none of the required information exists in the context, reply exactly:
"I don't know based on the provided documents."

Do not make up information.

Keep the answer concise and accurate.
                                          
Context:
{context}

Question:
{question}







Answer:
""")

In [ ]:
question="what is minimum cgpa required " 


# question=""

In [ ]:
response=multi_query_retriever.invoke(question)

In [ ]:
print(response[0].page_content)
print(response[0].metadata)

In [ ]:
context="\n=========\n".join(doc.page_content for doc in response)

In [ ]:
print(context)

In [ ]:
final_prompt = prompt.invoke(
    {
        "context": context,
        "question": question
    }
)

In [ ]:
print(final_prompt.messages[0].content)

In [ ]:
answer = llm.invoke(final_prompt)

In [ ]:
print(answer.content)

## EVALUATION

In [ ]:
## this code is upload the cvs file containing the question id and question and expected answers

import pandas as pd
 df=pd.read_csv(r"C:\Users\agarw\OneDrive\Desktop\RAG project\evaluation2.csv")
 df.head(5)

In [ ]:
 noW I WANT to create a loop that sends all these 50 questions to my llm one by one and then save the generated answerr in a file

result=[]

for index, row in df.iterrows():
    Question = row["Question"]
    Expected_Answer = row["Expected_Answer"]
    response=multi_query_retriever.invoke(Question)
    context="\n\n".join(doc.page_content for doc in response)
    final_prompt =prompt.invoke({
        "context":context,
        "question":Question
    })
    generated_answer=llm.invoke(final_prompt)
    generated_answer=generated_answer.content
    result.append({
        "Question_ID": row["Question_ID"],
        "Question":Question,
        "Expected_Answer":Expected_Answer,
        "generated_answer":generated_answer,
        "Retrived_Contents":context
    })
    
    
    
 

In [ ]:
# evaluation_df=pd.DataFrame(result)

In [ ]:
# evaluation_df.head(51)

In [ ]:
# evaluation_df.to_csv(
#     r"C:\Users\agarw\OneDrive\Desktop\RAG project\evaluation_answer.csv",
#     index=False
# )

# print("Evaluation dataset created successfully!")


In [ ]:
df=pd.read_csv(r"C:\Users\agarw\OneDrive\Desktop\RAG project\evaluation_answer.csv")
df.head(5)

In [ ]:
# Using BERT-Score for semantic Evaluation

import pandas as pd
from bert_score import score


In [ ]:
df=pd.read_csv(r"C:\Users\agarw\OneDrive\Desktop\RAG project\evaluation_answer.csv")

In [ ]:
references = df["Expected_Answer"].tolist()
candidates = df["generated_answer"].tolist()

In [ ]:
P,R,F1=score(candidates,references,lang="en",verbose=True)

In [ ]:
print(R.mean())
print(F1.mean())
print(P.mean())

In [ ]:
references = df["Expected_Answer"].tolist()
candidates = df["Retrived_Contents"].tolist()

In [ ]:
P,R,F1=score(candidates,references,lang="en",verbose=True)

In [ ]:
print(R.mean())
print(F1.mean())
print(P.mean())